# 05 · Live stress-test — 10 rounds, logged for analysis

A `TARGET` switch (cell 2) picks the race:
- **`'news'`** — competition 5, the live-web RAG pipeline (Guardian API + headless-Chromium bodies).
- **`'math'`** — competition 3, the `pipeline_maths` recipe from notebook 03 (adaptive `cot_v2` /
  `structured_enumeration_cot` routing, **no** retrieval, **no** calculator, `max_new_tokens=450`).

Plays the chosen competition **10 times back to back**, each round its own logged run
(`{target}_r01`..`{target}_r10`), then aggregates everything — per-round accuracy + reached level, and
**every wrong question with its diagnostics** (News: the retrieved evidence; Math: the reasoning chain,
which routing strategy fired, and whether it hit the token cap). The output dir keys off `TARGET`
(`experiments/news_test/` or `experiments/math_test/`).

> Leaderboard attempts are FREE (only the cumulative best counts), so re-running 10 rounds costs us
> nothing on the board. We still pause politely between games (the PDF asks: no rapid requests).
> The RAG-vs-noRAG ablation (last section) is **News-only**.

## 1 · Setup — clone/sync the repo, paths, the provided client

In [27]:
# Auto-reload edited src modules on every cell run -- so after a `git pull` the newest code lands without a
# manual importlib.reload or a restart. (Re-run the cell that USES the code, e.g. code-wire.)
# Colab's IPython ships an autoreload that does `from imp import reload`, and `imp` is GONE in Python 3.12 --
# so a tiny `imp` shim (reload only) we install first, then load the extension. BEST-EFFORT: any failure
# caught, so the cell never stalls (fall back: after a src pull, Runtime > Restart to pick changes up).
try:
    import sys as _sys, types as _types, importlib as _importlib
    if 'imp' not in _sys.modules:
        _imp = _types.ModuleType('imp')
        _imp.reload = _importlib.reload          # the one thing the old autoreload.py wants from `imp`.
        _sys.modules['imp'] = _imp
    _ip = get_ipython()
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
    print('autoreload: ON (src edits hot-reload on cell re-run)')
except Exception as _e:
    print(f'autoreload OFF ({type(_e).__name__}: {_e}) -- after a src pull, Runtime > Restart to pick changes up.')

import os, sys

REPO_URL = 'https://github.com/SleepyEveryD/NLP.git'
REPO_ROOT = '/content/NLP'
BRANCH = 'maths'
if not os.path.exists(REPO_ROOT):
  !git clone -b {BRANCH} {REPO_URL} {REPO_ROOT}
else:
  # Already cloned -> HARD-SYNC to the latest pushed branch (fetch + force-reset to origin/{BRANCH}).
  # Tracked files are overwritten to match remote; UNTRACKED run outputs are KEPT (experiments/runs/* is
  # gitignored). NOTE: `git pull` updates the FILES on disk -- it does NOT refresh THIS notebook's cells.
  !cd {REPO_ROOT} && git fetch -q origin && git checkout -q -f -B {BRANCH} origin/{BRANCH}

!cd {REPO_ROOT} && echo "on branch:" $(git rev-parse --abbrev-ref HEAD) "@" $(git --no-pager log -1 --oneline)

SRC = os.path.join(REPO_ROOT, 'src')
API_CLIENT = os.path.join(REPO_ROOT, 'NLP_assignment_api_client')
for p in (SRC, API_CLIENT):
  if p not in sys.path:
    sys.path.insert(0, p)
os.chdir(REPO_ROOT)
print('Repo root:', REPO_ROOT)

from millionaire_client import MillionaireClient
print('millionaire_client, imported it is.')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
autoreload: ON (src edits hot-reload on cell re-run)
on branch: maths @ 8d5ed72 Update 05_news_test.ipynb
Repo root: /content/NLP
millionaire_client, imported it is.


In [28]:
# The inference stack + the client's `requests`, install we do (light it stays). `-U` kept (Colab a stale
# bitsandbytes preinstalls); pandas/requests PINNED to Colab's versions (bare `-U` breaks google-colab/cudf).
!pip install -q -U 'transformers>=4.45.0' 'accelerate>=0.34.0' 'bitsandbytes>=0.46.1' sentencepiece einops pyyaml 'pandas==2.2.2' matplotlib 'requests==2.32.4'
print('Installed, the dependencies are.')

Installed, the dependencies are.


In [29]:
# Headless Chromium -- the live-NEWS body fetch it powers (configs/live.yaml: news_body_mode "browser").
# The relevance gate now ROUTES off-topic Guardian results here, so the browser matters MORE for News.
# Skip this only if you set retrieval.news_body_mode: "off".
!pip install -q playwright
!playwright install chromium
!playwright install-deps

# ARMED? a REAL launch the surest test is. NOT ready -> News falls back to HEADLINES only (crash-safe).
try:
    from playwright.sync_api import sync_playwright
    with sync_playwright() as _p:
        _b = _p.chromium.launch(headless=True); _b.close()
    print('headless Chromium: READY -- live-News body fetch armed.')
except Exception as _e:
    print(f'headless Chromium NOT ready ({type(_e).__name__}: {_e})')
    print('   -> News will use HEADLINES only. Re-run this cell, or set retrieval.news_body_mode: "off".')

Installing dependencies...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entr

## 2 · Config — pick the race (`TARGET`) + how many rounds

In [30]:
from config import RunConfig

config = RunConfig.from_yaml(os.path.join(REPO_ROOT, 'configs', 'live.yaml'))

# --- Pick which race to stress-test -------------------------------------------------------------
TARGET = 'entertainment'   # 'entertainment' (comp 0, knowledge RAG) | 'news' (comp 5, live-web RAG) | 'math' (comp 3, cot_v2 routing, NO RAG/tools)
# ------------------------------------------------------------------------------------------------

COMP_IDS = {'entertainment': 0, 'news': 5, 'math': 3}   # the competition id each target plays (confirm in the post-login list).
# Targets that use a retriever (knowledge/web RAG) -> the retrieval-flavoured diagnostics below key off this.
RETRIEVAL_TARGET = TARGET in ('entertainment', 'news')
COMP_ID  = COMP_IDS[TARGET]
NUM_ROUNDS = 10            # how many live games to play, back to back.
PAUSE_S    = 8.0          # polite gap between games (PDF: no rapid consecutive requests).

config.game.competition_id = COMP_ID
config.game.game_mode = 'text'

# The Guardian Open Platform key -- from a Colab secret (NEVER hardcoded). Only News uses it, but harmless
# to set always. With it, the Guardian fast body path is armed; the relevance gate keeps it ONLY on-topic.
try:
    from google.colab import userdata as _ud
    config.retrieval.guardian_api_key = _ud.get('guardian_key') or ''
except Exception:
    config.retrieval.guardian_api_key = config.retrieval.guardian_api_key or ''

print('TARGET:', TARGET, '| competition_id:', COMP_ID, f'({TARGET})')
print('rounds:', NUM_ROUNDS, '| pause between:', PAUSE_S, 's')
print('aim_seconds:', config.game.aim_seconds, '| model:', config.model.name, '|', config.model.quantization)
if TARGET == 'news':
    print('RAG:', 'ON' if config.retrieval.enabled else 'OFF', '| source:', config.retrieval.source,
          '| news_body_mode:', config.retrieval.news_body_mode, '| fetch_bodies:', config.retrieval.news_fetch_bodies)
    print('Guardian API:', 'KEY SET' if config.retrieval.guardian_api_key else 'no key -> News uses browser')
elif TARGET == 'entertainment':
    print('Entertainment pipeline: few_shot_entertainment + RAG (routed -> FAISS/Wikipedia, gated by'
          ' needs_retrieval). Pop-culture recall, no chain-of-thought.')
    print('RAG:', 'ON' if config.retrieval.enabled else 'OFF', '| source:', config.retrieval.source, '| top_k:', config.retrieval.top_k)
else:  # math
    print('Maths pipeline: adaptive cot_v2 / structured_enumeration_cot routing, NO retrieval, NO calculator,'
          ' max_new_tokens=450 (post-B3, 30s-wall safe).')

TARGET: math | competition_id: 3 (math)
rounds: 10 | pause between: 8.0 s
aim_seconds: 25.0 | model: Qwen/Qwen2.5-7B-Instruct | 4bit
Maths pipeline: adaptive cot_v2 / structured_enumeration_cot routing, NO retrieval, NO calculator, max_new_tokens=450 (post-B3, 30s-wall safe).


## 3 · Load + warm up the model

In [31]:
import time
from inference.engine import TransformersEngine

t0 = time.perf_counter()
if 'engine' not in globals():
      engine = TransformersEngine(model_name=config.model.name,
                                  quantization=config.model.quantization,
                                  dtype=config.model.dtype)
      engine.warmup()
else:
      print('engine 已在显存中,跳过加载。')
print(f'Model loaded in {time.perf_counter() - t0:.1f}s')

t0 = time.perf_counter()
engine.warmup()
print(f'Warmup in {time.perf_counter() - t0:.1f}s')

engine 已在显存中,跳过加载。
Model loaded in 0.0s
Warmup in 0.6s


## 4 · Wire the pipeline for `TARGET` + log in to the game

In [32]:
from classify.classifier import QuestionClassifier
from prompting.builder import PromptBuilder, RoutingPromptBuilder
from classify.reasoning_router import MATHS_LIVE_POLICY, MATHS_LIVE_FALLBACK
from agent.pipeline import QAPipeline
from tools import default_tools, solve_maths
from retrieval import build_retriever

if RETRIEVAL_TARGET:
    # Phase 4 RAG: the routing retriever. News (post-cutoff) `routed` sends questions to the live web
    # (Guardian API + relevance gate -> headless-Chromium on the gnews link); Entertainment routes the
    # SAME `routed` retriever to FAISS/Wikipedia (topic != News). `needs_retrieval` gates it per question.
    retriever = build_retriever(config.retrieval)
    print('RAG:', (f'ON  source={config.retrieval.source}  top_k={config.retrieval.top_k}') if retriever else 'OFF')

    # Per-race strategy: Entertainment gets its OWN `few_shot_entertainment` (pop-culture exemplars +
    # domain instruction, mirrors notebook 03's pipeline_entertainment); News keeps config.prompt_strategy
    # (few_shot_v1). Both: RAG + the classifier-gated calculator no-op.
    strategy = 'few_shot_entertainment' if TARGET == 'entertainment' else config.prompt_strategy
    pipeline = QAPipeline(
        engine=engine,
        prompt_builder=PromptBuilder(strategy=strategy),
        classifier=QuestionClassifier(),
        retriever=retriever,
        tools=default_tools(),
        latency_budget_s=config.latency_budget_s,
    )
    print(f'{TARGET.capitalize()} pipeline wired:', strategy, '+ RAG (routed) + classifier-gated tools')

else:  # math -- EXACTLY the `pipeline_maths` recipe from notebook 03 (comp 3).
    # Adaptive routing: counting / temporal / discrete-enumeration -> structured_enumeration_cot; everything
    # else (arithmetic, logic, concept/stats) -> the cot_v2 fallback. NO retrieval (it only distracts on
    # Maths), NO calculator (at n=1 it clobbers the chain on numeric Qs), single-pass. max_new_tokens=450
    # (raised from 300 post-B3: only short time-interval Qs reach structured enum now, so chains finish in
    # 5-10s -- big headroom; 450 tok ~= 28s worst-case, dial to 400 if any Maths turn nears the 30s wall).
    pipeline = QAPipeline(
        engine=engine,
        prompt_builder=RoutingPromptBuilder(policy=MATHS_LIVE_POLICY, fallback_strategy=MATHS_LIVE_FALLBACK),
        classifier=QuestionClassifier(),
        retriever=None,
        tools=None,
        solver=solve_maths,   # deterministic type-specific solver: short-circuits the LLM on solvable types.
        latency_budget_s=config.latency_budget_s,
        max_new_tokens=450,
    )
    print('Maths pipeline wired: ADAPTIVE (counting/temporal/enum -> structured_enumeration_cot; else -> cot_v2)'
          ' + 450 tokens + single-pass + NO retrieval + NO general-calculator + DETERMINISTIC solver (solve_maths)')

# --- Log in to the real game ---
from google.colab import userdata
from game.client import GameClient

USERNAME = userdata.get('username')
PASSWORD = userdata.get('password')

game_client = GameClient()
game_client.login(USERNAME, PASSWORD)
print('Logged in as', USERNAME)

# The competitions + ids (safe -- starts no timer). Confirm the target's id here (News=5, Maths=3).
for c in game_client.list_competitions():
    print('  id=', c.id, '|', c.name, '| max_levels=', getattr(c, 'max_levels', '?'))

Maths pipeline wired: ADAPTIVE (counting/temporal/enum -> structured_enumeration_cot; else -> cot_v2) + 450 tokens + single-pass + NO retrieval + NO general-calculator + DETERMINISTIC solver (solve_maths)
Logged in as runjie dai
  id= 0 | Entertainment | max_levels= 15
  id= 1 | Ancient History and Politics | max_levels= 15
  id= 2 | Science and Nature | max_levels= 15
  id= 3 | Maths | max_levels= 15
  id= 4 | Philosophy and Psychology | max_levels= 15
  id= 5 | News | max_levels= 15


## 5 · ▶ Play 10 live rounds  (each its own logged run)

Plays the `TARGET` game `NUM_ROUNDS` times. Each round writes its own run dir `{target}_r{NN}` (so no
round overwrites another — the LiveRunner truncates *within* a run_id). One round failing (a rate-limit, a
network blip) is caught and logged as a gap — the loop carries on.

In [33]:
import time
from evaluation.runner import run_session

LOG_ROOT = os.path.join(REPO_ROOT, 'experiments', 'runs')
round_runs = []   # [(round_no, run_path-or-None)]

for r in range(1, NUM_ROUNDS + 1):
    config.run_id = f'{TARGET}_r{r:02d}'
    print(f'\n===== ▶ ROUND {r}/{NUM_ROUNDS}  (run_id={config.run_id}) =====')
    try:
        path = run_session(pipeline, config, game_client=game_client, log_root=LOG_ROOT)
        round_runs.append((r, path))
        print('   round log:', path)
    except Exception as e:
        round_runs.append((r, None))
        print(f'   ⚠️ round {r} FAILED ({type(e).__name__}: {e}) -- logged as a gap, the loop continues.')
    if r < NUM_ROUNDS:
        time.sleep(PAUSE_S)   # polite gap between live games.

print('\nAll rounds done. Logged runs:', [p for _r, p in round_runs if p])


===== ▶ ROUND 1/10  (run_id=math_r01) =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=6785 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=11.9s (left was 29.91571)
[2] qid=6756 lvl=0 reached=1 -> B | correct=False | timed_out=False | latency=4.6s (left was 29.91551)
   round log: /content/NLP/experiments/runs/math_r01

===== ▶ ROUND 2/10  (run_id=math_r02) =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=6697 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=10.1s (left was 29.916317)
[2] qid=6927 lvl=0 reached=1 -> B | correct=False | timed_out=False | latency=8.1s (left was 29.915215)
   round log: /content/NLP/experiments/runs/math_r02

===== ▶ ROUND 3/10  (run_id=math_r03) =====


...

[1] qid=6745 lvl=0 reached=0 -> C | correct=False | timed_out=False | latency=4.4s (left was 29.911553)
   round log: /content/NLP/experiments/runs/math_r08

===== ▶ ROUND 9/10  (run_id=math_r09) =====
[1] qid=6728 lvl=0 reached=None -> B | correct=True | timed_out=False | latency=0.0s (left was 29.91278)


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[2] qid=6970 lvl=0 reached=1 -> D | correct=False | timed_out=False | latency=4.9s (left was 29.916662)
   round log: /content/NLP/experiments/runs/math_r09

===== ▶ ROUND 10/10  (run_id=math_r10) =====


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


[1] qid=6751 lvl=0 reached=0 -> D | correct=False | timed_out=False | latency=7.4s (left was 29.915911)
   round log: /content/NLP/experiments/runs/math_r10

All rounds done. Logged runs: ['/content/NLP/experiments/runs/math_r01', '/content/NLP/experiments/runs/math_r02', '/content/NLP/experiments/runs/math_r03', '/content/NLP/experiments/runs/math_r04', '/content/NLP/experiments/runs/math_r05', '/content/NLP/experiments/runs/math_r06', '/content/NLP/experiments/runs/math_r07', '/content/NLP/experiments/runs/math_r08', '/content/NLP/experiments/runs/math_r09', '/content/NLP/experiments/runs/math_r10']


## 6 · Analysis — per-round scores + every wrong question (with diagnostics)

Aggregates all rounds and **saves** the consolidated records to `experiments/{TARGET}_test/` so they ride
back to the repo. Three artifacts: a per-round summary, every question, and the wrong questions alone.
- **News**: wrong questions carry the retrieved evidence text (retrieval-miss vs grounding-miss diagnosis)
  and a retrieval source mix.
- **Math**: wrong questions carry the reasoning chain (`raw_output`), which routing strategy fired
  (`prompt_strategy`), and the output token count — so you can see truncation at the 300-token cap.

In [34]:
import json, collections
from pathlib import Path
import pandas as pd

OUT_DIR = Path(REPO_ROOT) / 'experiments' / f'{TARGET}_test'
OUT_DIR.mkdir(parents=True, exist_ok=True)

def _read_round(path):
    """One round's records.jsonl -> list of dict rows ([] if missing/empty)."""
    if not path:
        return []
    p = Path(path) / 'records.jsonl'
    if not p.exists():
        return []
    return [json.loads(l) for l in p.read_text(encoding='utf-8').splitlines() if l.strip()]

def _sources(row):
    """The retrieval SOURCE mix for a row, from retrieved_snippets ('[theguardian.com#..] ' prefix)."""
    out = collections.Counter()
    for s in (row.get('retrieved_snippets') or []):
        if isinstance(s, str) and s.startswith('['):
            out[s[1:].split('#', 1)[0].split(']', 1)[0]] += 1
    return dict(out)

# --- Gather every round ---
summary_rows, all_q, wrong_q = [], [], []
for r, path in round_runs:
    rows = _read_round(path)
    graded = [x for x in rows if x.get('correct') is not None]
    n_correct = sum(1 for x in graded if x.get('correct') is True)
    reached = [x.get('reached_level') for x in rows if x.get('reached_level') is not None]
    summary_rows.append({
        'round': r,
        'answered': len(rows),
        'correct': n_correct,
        'graded': len(graded),
        'accuracy': (n_correct / len(graded)) if graded else float('nan'),
        'reached_level': max(reached) if reached else None,
    })
    for x in rows:
        rec = {'round': r, **x, 'sources': _sources(x)}
        all_q.append(rec)
        if x.get('correct') is False:
            wrong_q.append(rec)

# --- Per-round summary + overall ---
summary = pd.DataFrame(summary_rows)
print(f'PER-ROUND SUMMARY ({TARGET})')
print(summary.to_string(index=False))
tot_c = int(summary['correct'].sum()); tot_g = int(summary['graded'].sum())
lv = [s['reached_level'] for s in summary_rows if s['reached_level'] is not None]
print(f"\nOVERALL: {tot_c}/{tot_g} graded = {tot_c / tot_g:.1%}" if tot_g else '\nOVERALL: no graded answers')
if lv:
    print(f"reached_level over {len(lv)} rounds: min={min(lv)} max={max(lv)} mean={sum(lv)/len(lv):.1f} | {sorted(lv, reverse=True)}")

if RETRIEVAL_TARGET:
    # --- Retrieval source mix across ALL questions (did the gate route to the browser? did docs land?) ---
    src_total = collections.Counter()
    for x in all_q:
        src_total.update(x['sources'])
    print('\nRETRIEVAL SOURCE MIX (doc count across all', len(all_q), 'questions):', dict(src_total))
    fired = sum(1 for x in all_q if x.get('retrieval_used'))
    print(f"retrieval fired on {fired}/{len(all_q)} questions")
else:  # math -- which routing strategy fired, and how close to the 300-token cap the chains ran.
    strat_mix = collections.Counter(x.get('prompt_strategy') or '?' for x in all_q)
    print('\nROUTING STRATEGY MIX (across all', len(all_q), 'questions):', dict(strat_mix))
    toks = [x.get('tokens_out', 0) for x in all_q if x.get('tokens_out')]
    if toks:
        capped = sum(1 for t in toks if t >= 300)
        print(f"tokens_out: min={min(toks)} max={max(toks)} mean={sum(toks)/len(toks):.0f}"
              f" | hit the 300 cap on {capped}/{len(toks)} questions (likely truncated before 'Answer:')")

PER-ROUND SUMMARY (math)
 round  answered  correct  graded  accuracy  reached_level
     1         2        1       2       0.5              1
     2         2        1       2       0.5              1
     3         2        1       2       0.5              1
     4         2        1       2       0.5              1
     5         1        0       1       0.0              0
     6         2        1       2       0.5              1
     7         5        4       5       0.8              4
     8         1        0       1       0.0              0
     9         2        1       2       0.5              1
    10         1        0       1       0.0              0

OVERALL: 10/20 graded = 50.0%
reached_level over 10 rounds: min=0 max=4 mean=1.0 | [4, 1, 1, 1, 1, 1, 1, 0, 0, 0]

ROUTING STRATEGY MIX (across all 20 questions): {'cot_v2': 18, 'structured_enumeration_cot': 2}
tokens_out: min=19 max=321 mean=67 | hit the 300 cap on 1/19 questions (likely truncated before 'Answer:')


In [35]:
# --- Every WRONG question, with the right diagnostics for the target ---
#   News / Entertainment: the retrieved EVIDENCE (retrieval-miss vs grounding-miss check).
#   Math: the reasoning chain (raw_output), the routing strategy that fired, and tokens_out (cap = 300).
print(f"{'=' * 78}\nEVERY WRONG QUESTION  ({len(wrong_q)} across {NUM_ROUNDS} rounds)\n{'=' * 78}")
for x in wrong_q:
    opts = x.get('options') or {}
    pick = x.get('predicted_answer')
    if RETRIEVAL_TARGET:
        print(f"\n[round {x['round']}] qid={x['qid']} reached_level={x.get('reached_level')} "
              f"lat={x.get('latency_s', 0):.1f}s sources={x['sources']}")
    else:
        toks = x.get('tokens_out', 0)
        print(f"\n[round {x['round']}] qid={x['qid']} level={x.get('level')} reached_level={x.get('reached_level')} "
              f"lat={x.get('latency_s', 0):.1f}s strategy={x.get('prompt_strategy')} "
              f"tokens_out={toks}{'  <-- HIT 300 CAP (truncated?)' if toks and toks >= 300 else ''}")
    print(f"Q: {x['question_text']}")
    for k, v in opts.items():
        print(f"   {k}. {v}" + ('  <-- our pick (WRONG)' if k == pick else ''))
    if RETRIEVAL_TARGET:
        snips = x.get('retrieved_snippets') or []
        if snips:
            print('   -- retrieved evidence --')
            for s in snips:
                print(f"      {str(s)[:500]}")
        else:
            print('   (no retrieved evidence logged)')
    else:  # math -- the model's reasoning chain (truncation / wrong set-up shows here).
        raw = (x.get('raw_output') or '').strip()
        print('   -- reasoning chain --')
        print('      ' + (raw[:900].replace('\n', '\n      ') if raw else '(empty)'))

# --- SAVE the consolidated artifacts (ride back to the repo via experiments/) ---
summary.to_csv(OUT_DIR / 'summary.csv', index=False)
with open(OUT_DIR / 'all_questions.jsonl', 'w', encoding='utf-8') as f:
    for x in all_q:
        f.write(json.dumps(x, ensure_ascii=False) + '\n')
with open(OUT_DIR / 'wrong_questions.jsonl', 'w', encoding='utf-8') as f:
    for x in wrong_q:
        f.write(json.dumps(x, ensure_ascii=False) + '\n')
print(f"\nSaved -> {OUT_DIR}/  (summary.csv, all_questions.jsonl [{len(all_q)}], wrong_questions.jsonl [{len(wrong_q)}])")

EVERY WRONG QUESTION  (10 across 10 rounds)

[round 1] qid=6756 level=0 reached_level=1 lat=4.6s strategy=cot_v2 tokens_out=32
Q: A kidney dialysis center periodically checks a sample of its equipment and performs a major recalibration if readings are sufficiently off target. Similarly, a fabric factory periodically checks the sizes of towels coming off an assembly line and halts production if measurements are sufficiently off target. In both situations, we have the null hypothesis that the equipment is performing satisfactorily. For each situation, which is the more serious concern, a Type I or Type II error?
   A. Dialysis center: Type II error, towel manufacturer: Type II error
   B. Dialysis center: Type I error, towel manufacturer: Type II error  <-- our pick (WRONG)
   C. Dialysis center: Type I error, towel manufacturer: Type I error
   D. Dialysis center: Type II error, towel manufacturer: Type I error
   -- reasoning chain --
      Step 1: Dialysis center needs to avoid missin

## 8 · RAG vs no-RAG ablation — retrieval targets only — do questions answer better WITHOUT retrieval?

**Runs only for retrieval targets (`TARGET` in `entertainment` / `news`)** — Maths uses no retrieval, so there is nothing to ablate, and this cell is skipped there.

Some News questions are really **knowledge/historical** ("which US president visited China in 2008..") —
the model may know the answer from its **own training**, and an off-topic retrieved article can *mislead*
it (grounding on junk). This re-answers the SAME questions **with** retrieval and **without**, side by
side, so we can see where RAG helps vs hurts. Annotate `KNOWN_GOLD` for questions you can verify by hand
to get a score (live games hide the gold).

In [36]:
if not RETRIEVAL_TARGET:
    print(f"RAG-vs-noRAG ablation needs a retriever (Maths uses none) -- skipping for TARGET = {TARGET}.")
else:
    import json
    from schemas import Question, QuestionType
    from agent.pipeline import QAPipeline
    from prompting.builder import PromptBuilder
    from classify.classifier import QuestionClassifier
    from tools import default_tools

    # Twin of THIS target's RAG pipeline, but retriever=None -> pure parametric knowledge (no RAG).
    _strategy = 'few_shot_entertainment' if TARGET == 'entertainment' else config.prompt_strategy
    pipeline_norag = QAPipeline(
        engine=engine,
        prompt_builder=PromptBuilder(strategy=_strategy),
        classifier=QuestionClassifier(),
        retriever=None,                 # <- the only difference from the RAG `pipeline`.
        tools=default_tools(),
        latency_budget_s=config.latency_budget_s,
    )

    # Questions to probe. Default: the wrong questions THIS News run logged. Point SRC elsewhere to test others.
    SRC = os.path.join(REPO_ROOT, 'experiments', f'{TARGET}_test', 'wrong_questions.jsonl')
    rows = [json.loads(l) for l in open(SRC, encoding='utf-8') if l.strip()]

    # Known gold BY TEXT substring (resolved to the option letter at runtime -> robust to option shuffling).
    # Fill in the ones you can verify; un-annotated rows are still printed for eyeballing.
    # NOTE: the entries below are News examples -- for Entertainment, replace them with your own verified golds.
    KNOWN_GOLD = {
        '10851': 'George W. Bush',     # Bush attended a Beijing church service, 2008 Olympics
        '10659': 'CEPI',               # Coalition for Epidemic Preparedness Innovations
        '12017': 'Cannes',             # Cannes Film Festival opens ~May 12
        '10645': '161',                # Pentagon released ~16x declassified UFO files
        '10747': '1.5 million',        # Labour's housing pledge
    }

    def _build_q(r):
        try: qt = QuestionType(r.get('qtype', 'mcq'))
        except Exception: qt = QuestionType.MCQ
        return Question(qid=r['qid'], text=r['question_text'], options=r.get('options') or {},
                        qtype=qt, level=r.get('level'), topic=r.get('topic'), language=r.get('language'))

    def _gold_letter(r):
        sub = KNOWN_GOLD.get(str(r['qid']))
        if not sub:
            return None
        for k, v in (r.get('options') or {}).items():
            if sub.lower() in str(v).lower():
                return k
        return None

    print(f"{'qid':>7} | RAG | noRAG | gold | verdict")
    print('-' * 70)
    rag_ok = norag_ok = scored = 0
    for r in rows:
        q = _build_q(r)
        a_rag = pipeline.answer(q).answer          # WITH live retrieval
        a_no  = pipeline_norag.answer(q).answer    # parametric knowledge only
        gold = _gold_letter(r)
        verdict = ''
        if gold:
            scored += 1
            rag_ok   += (a_rag == gold)
            norag_ok += (a_no  == gold)
            verdict = f"RAG {'OK' if a_rag==gold else 'X'} | noRAG {'OK' if a_no==gold else 'X'}"
        print(f"{r['qid']:>7} |  {a_rag}   |  {a_no}    |  {gold or '-'}   | {verdict}")
        print(f"          Q: {r['question_text'][:82]}")

    if scored:
        print(f"\nON {scored} ANNOTATED-GOLD QUESTIONS:   RAG {rag_ok}/{scored}    no-RAG {norag_ok}/{scored}")
    print("\n(Eyeball RAG vs noRAG on the un-annotated rows; add to KNOWN_GOLD to score more.)")

RAG-vs-noRAG ablation is News-only (Maths uses no retrieval) -- skipping for TARGET = math.


## 9 · Solver self-test (Maths-only) — confirm the deterministic solver is wired & firing

**Runs only when `TARGET == 'math'`** — the solver is a Maths component; skipped for entertainment / news.

Doesn't depend on luck-of-the-draw. **Part 1** calls `solve_maths()` directly (pure Python, no model). **Part 2** runs the same questions through a Maths pipeline with `solver=solve_maths` — if `tool_used=math_solver` and `tokens_out=0`, the LLM was short-circuited (the program answered, not the model).

In [37]:
# Maths-only: the deterministic solver is a Maths component -- skip it on other targets so an
# entertainment / news run never executes the Maths self-test.
if TARGET != 'math':
    print(f"Solver self-test is Maths-only -- skipping for TARGET = {TARGET}.")
else:
    # ============================================================================
    # Solver self-test -- confirm the deterministic solver is WIRED and FIRING (no luck-of-the-draw).
    # Part 1: call solve_maths() directly (pure Python, NO model) -> proves the program computes the answer.
    # Part 2: run the SAME questions through a Maths pipeline with solver=solve_maths -> proves the LLM is
    #         SHORT-CIRCUITED (tool_used='math_solver', tokens_out=0 -> the model never generated anything).
    # Run this BEFORE the model-comparison cell below (that one frees `engine`).
    # ============================================================================
    from schemas import Question, QuestionType
    from tools import solve_maths
    from prompting.builder import RoutingPromptBuilder
    from classify.reasoning_router import MATHS_LIVE_POLICY, MATHS_LIVE_FALLBACK
    from classify.classifier import QuestionClassifier
    from agent.pipeline import QAPipeline

    _TESTS = [
        ('6886', 'Find all zeros in the indicated finite field of the given polynomial x^5 + 3x^3 + x^2 + 2x in Z_5',
         {'A': '0,4', 'B': '0,1', 'C': '0', 'D': '1'}, 'A'),
        ('6722', 'Find the characteristic of the ring Z_3 x Z_3', {'A': '3', 'B': '30', 'C': '12', 'D': '0'}, 'A'),
        ('6962', 'Two numbers added together are 19. Their product is 70. What are the two numbers?',
         {'A': '7, 10', 'B': '4, 15', 'C': '5, 14', 'D': '3, 16'}, 'C'),
        ('6809', 'What is the greatest common divisor of $2^{1001}-1$ and $2^{1012}-1$?',
         {'A': '2047', 'B': '2049', 'C': '1', 'D': '2048'}, 'A'),
    ]

    print("PART 1 -- solve_maths() directly (pure Python, no model):")
    for qid, text, opts, gold in _TESTS:
        q = Question(qid=qid, text=text, options=opts, qtype=QuestionType.MCQ)
        r = solve_maths(q)
        ok = '✅' if (r and r[0] == gold) else '❌'
        print(f"  {qid}: {r}  expect={gold}  {ok}")

    print("\nPART 2 -- through a Maths pipeline (solver=solve_maths) -> the model should be SKIPPED:")
    _mp = QAPipeline(engine=engine,
                     prompt_builder=RoutingPromptBuilder(policy=MATHS_LIVE_POLICY, fallback_strategy=MATHS_LIVE_FALLBACK),
                     classifier=QuestionClassifier(), retriever=None, tools=None, solver=solve_maths,
                     latency_budget_s=config.latency_budget_s, max_new_tokens=450)
    print(f"{'qid':>6} | tool_used   | tokens_out | answer | expect | ok")
    print('-' * 62)
    for qid, text, opts, gold in _TESTS:
        p = _mp.answer(Question(qid=qid, text=text, options=opts, qtype=QuestionType.MCQ))
        ok = '✅' if p.answer == gold else '❌'
        print(f"{qid:>6} | {str(p.tool_used):11} | {p.tokens_out:>10} | {p.answer:^6} | {gold:^6} | {ok}")
    print("\nIf tool_used=math_solver AND tokens_out=0 -> the solver is LIVE and the model was NOT used on those.")
    print("If tool_used=None here, the solver is NOT wired in this notebook's pipeline (re-pull / re-run cell 10).")


PART 1 -- solve_maths() directly (pure Python, no model):
  6886: ('A', 'finite-field roots in Z_5: [0, 4]')  expect=A  ✅
  6722: ('A', 'ring characteristic = lcm(3, 3) = 3')  expect=A  ✅
  6962: ('C', 'two numbers with sum 19, product 70: [5, 14]')  expect=C  ✅
  6809: ('A', 'gcd(2^1001-1, 2^1012-1) = 2^11-1 = 2047')  expect=A  ✅

PART 2 -- through a Maths pipeline (solver=solve_maths) -> the model should be SKIPPED:
   qid | tool_used   | tokens_out | answer | expect | ok
--------------------------------------------------------------
  6886 | math_solver |          0 |   A    |   A    | ✅
  6722 | math_solver |          0 |   A    |   A    | ✅
  6962 | math_solver |          0 |   C    |   C    | ✅
  6809 | math_solver |          0 |   A    |   A    | ✅

If tool_used=math_solver AND tokens_out=0 -> the solver is LIVE and the model was NOT used on those.
If tool_used=None here, the solver is NOT wired in this notebook's pipeline (re-pull / re-run cell 10).


## 10 · Model comparison (Maths-only) — base 7B vs a math-specialised open model

**Runs only when `TARGET == 'math'`.** ⚠️ It loads a SECOND model and FREES `engine` — never run it on an entertainment / news session (it would tear down the live pipeline).

Answers the assignment's *"are certain models better at certain topics?"*. Same Maths questions through both models' **pure-LLM** pipeline (solver OFF — measures the *model*), scored vs hand-verified gold, with flips and >25s flags.

> Open-weight + local only (no API). On a **T4** the two 7B-4bit models can't co-reside, so the base model is freed before the math model loads — **re-run the *Load + warm up the model* cell afterwards** to restore `engine`. ⚠️ This cell must be RUN on Colab (loads a 2nd model, ~5 min); it does nothing if just pulled.

In [38]:
# Maths-only AND destructive (loads a 2nd model, FREES `engine`). Guard hard: on any non-Maths
# target skip entirely, so an entertainment / news run never tears down its own live engine.
if TARGET != 'math':
    print(f"Model comparison is Maths-only (and frees `engine`) -- skipping for TARGET = {TARGET}.")
else:
    # ============================================================================
    # 9 · MODEL COMPARISON (Maths) -- base Qwen2.5-7B vs a math-specialised open model.
    # Assignment investigation question: "are certain models better at certain topics?"
    # Same Maths questions through BOTH models' PURE-LLM pipeline (solver OFF -> we measure the
    # MODEL, not the deterministic short-circuit), scored vs hand-verified gold, flags flips + >25s.
    # Rules: open-weight, local only (no API). T4 NOTE: two 7B-4bit can't co-reside, so we FREE the
    # base model before loading the math one -> re-run the "Load + warm up the model" cell to restore it.
    # ============================================================================
    import time, json, gc, os
    import torch
    from schemas import Question, QuestionType
    from prompting.builder import RoutingPromptBuilder
    from classify.reasoning_router import MATHS_LIVE_POLICY, MATHS_LIVE_FALLBACK
    from classify.classifier import QuestionClassifier
    from agent.pipeline import QAPipeline
    from inference.engine import TransformersEngine

    MATH_MODEL = 'Qwen/Qwen2.5-Math-7B-Instruct'   # the challenger (open-weight only); swap to try others.
    FREE_A_BEFORE_B = True                          # T4: free the base model before loading the math one.

    # --- question set: this run's saved Maths questions if present, else a small embedded fallback ---
    rows, SRC = [], None
    for cand in ('all_questions.jsonl', 'wrong_questions.jsonl'):
        p = os.path.join(REPO_ROOT, 'experiments', 'math_test', cand)
        if os.path.exists(p):
            SRC = p
            rows = [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]
            break
    if not rows:   # fallback so the cell always runs (knowledge-heavy: where a math model should help)
        SRC = 'embedded fallback'
        rows = [
            {'qid': '6919', 'question_text': 'Statement 1 | Q is an extension field of Z_2. Statement 2 | Every non-constant polynomial over a field has a zero in some extension field.', 'options': {'A': 'True, False', 'B': 'False, False', 'C': 'True, True', 'D': 'False, True'}},
            {'qid': '6713', 'question_text': 'Statement 1 | If H is a subgroup of a group G and a belongs to G, then aH = Ha. Statement 2 | If H is normal of G and a belongs to G, then ah = ha for all h in H.', 'options': {'A': 'True, False', 'B': 'True, True', 'C': 'False, False', 'D': 'False, True'}},
            {'qid': '6910', 'question_text': 'Statement 1 | If T: V -> W is a linear transformation and dim(V) < dim(W) < 1, then T must be injective. Statement 2 | Let dim(V) = n and suppose that T: V -> V is linear. If T is injective, then it is a bijection.', 'options': {'A': 'False, False', 'B': 'False, True', 'C': 'True, True', 'D': 'True, False'}},
            {'qid': '6767', 'question_text': 'Find the order of the factor group (Z_4 x Z_12)/(<2> x <2>)', 'options': {'A': '3', 'B': '4', 'C': '12', 'D': '2'}},
            {'qid': '6886', 'question_text': 'Find all zeros in the indicated finite field of the given polynomial with coefficients in that field. x^5 + 3x^3 + x^2 + 2x in Z_5', 'options': {'A': '0,4', 'B': '0,1', 'C': '0', 'D': '1'}},
        ]
    rows = list({r['qid']: r for r in rows}.values())   # dedupe by qid
    print(f"comparison set: {len(rows)} Maths questions  (source: {SRC})")

    # --- hand-verified gold BY TEXT SUBSTRING (unambiguous within each option set). Add more freely. ---
    KNOWN_GOLD = {
        '6886': '0,4', '6809': '2047', '6962': '5, 14', '6728': '37', '6932': '8', '7021': '0', '6841': '46',
        '6899': 'Friday', '6817': '70', '6767': '4', '6781': '3', '6671': '24', '6688': '15',
        '6919': 'False, True', '6713': 'False, False', '6910': 'True, True', '6912': 'True, True',
        '6894': 'False, False', '6696': 'False, True',
    }

    def _bq(r):
        try: qt = QuestionType(r.get('qtype', 'mcq'))
        except Exception: qt = QuestionType.MCQ
        return Question(qid=r['qid'], text=r['question_text'], options=r.get('options') or {}, qtype=qt,
                        level=r.get('level'), topic=r.get('topic'), language=r.get('language'))

    def _gold(r):
        sub = KNOWN_GOLD.get(str(r['qid']))
        if not sub: return None
        hits = [k for k, v in (r.get('options') or {}).items() if sub.lower() in str(v).lower()]
        return hits[0] if len(hits) == 1 else None

    def _make_pipe(eng):   # PURE-LLM Maths pipeline (solver OFF) -> measures the MODEL itself.
        return QAPipeline(engine=eng, prompt_builder=RoutingPromptBuilder(policy=MATHS_LIVE_POLICY, fallback_strategy=MATHS_LIVE_FALLBACK),
                          classifier=QuestionClassifier(), retriever=None, tools=None, solver=None,
                          latency_budget_s=config.latency_budget_s, max_new_tokens=450)

    def _run(eng, label):
        pipe, out = _make_pipe(eng), {}
        for r in rows:
            t0 = time.perf_counter(); pred = pipe.answer(_bq(r)); out[r['qid']] = (pred.answer, time.perf_counter() - t0)
        print(f"  {label}: {len(out)} answered"); return out

    print(f"\n[A] base model: {config.model.name}")
    resA = _run(engine, 'A')

    if FREE_A_BEFORE_B:
        del engine; gc.collect(); torch.cuda.empty_cache()
        print("   freed base model (re-run the model-load cell to restore `engine` afterwards).")

    print(f"\n[B] loading math model: {MATH_MODEL}")
    engine_math = TransformersEngine(model_name=MATH_MODEL, quantization=config.model.quantization, dtype=config.model.dtype)
    engine_math.warmup()
    resB = _run(engine_math, 'B')

    # --- compare ---
    print(f"\n{'qid':>7} | A | B | gold | note")
    print('-' * 64)
    a_ok = b_ok = scored = Bfix = Bbreak = 0; over = []
    for r in rows:
        q = r['qid']; (aA, latA), (aB, latB) = resA[q], resB[q]; g = _gold(r)
        if latA > 25: over.append((q, 'A', round(latA, 1)))
        if latB > 25: over.append((q, 'B', round(latB, 1)))
        note = ''
        if g:
            scored += 1; oa, ob = (aA == g), (aB == g); a_ok += oa; b_ok += ob
            if ob and not oa: Bfix += 1; note = 'B fixes A'
            elif oa and not ob: Bbreak += 1; note = 'B BREAKS A'
        print(f"{q:>7} | {aA} | {aB} | {g or '-'} | {note}{'  <-DIFF' if aA != aB else ''}")

    if scored:
        print(f"\nON {scored} GOLD-LABELLED:  base-A {a_ok}/{scored} ({a_ok/scored:.0%})   |   math-B {b_ok}/{scored} ({b_ok/scored:.0%})")
        print(f"   B fixes A: {Bfix}   |   B breaks A: {Bbreak}   (net {Bfix-Bbreak:+d})")
    print(f"latency >25s (wall risk): {over or 'none'}")
    print("\nNOTE: this measures the MODEL (solver OFF). Add qids to KNOWN_GOLD to score more.")
    print("To restore the 7B for other cells, RE-RUN the 'Load + warm up the model' cell.")


comparison set: 20 Maths questions  (source: /content/NLP/experiments/math_test/all_questions.jsonl)

[A] base model: Qwen/Qwen2.5-7B-Instruct


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  A: 20 answered
   freed base model (re-run the model-load cell to restore `engine` afterwards).

[B] loading math model: Qwen/Qwen2.5-Math-7B-Instruct


config.json:   0%|          | 0.00/658 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.32k [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/161 [00:00<?, ?B/s]

  B: 20 answered

    qid | A | B | gold | note
----------------------------------------------------------------
   6785 | B | A | - |   <-DIFF
   6756 | B | A | - |   <-DIFF
   6697 | B | A | - |   <-DIFF
   6927 | B | D | - |   <-DIFF
   7027 | B | D | - |   <-DIFF
   6976 | B | A | - |   <-DIFF
   6744 | A | A | - | 
   6688 | B | C | D |   <-DIFF
   6761 | C | A | - |   <-DIFF
   6710 | A | A | - | 
   6913 | B | A | - |   <-DIFF
   6958 | C | A | - |   <-DIFF
   6644 | D | A | - |   <-DIFF
   6877 | C | A | - |   <-DIFF
   6878 | D | A | - |   <-DIFF
   7013 | A | C | - |   <-DIFF
   6745 | C | A | - |   <-DIFF
   6728 | C | B | B | B fixes A  <-DIFF
   6970 | D | A | - |   <-DIFF
   6751 | D | A | - |   <-DIFF

ON 2 GOLD-LABELLED:  base-A 0/2 (0%)   |   math-B 1/2 (50%)
   B fixes A: 1   |   B breaks A: 0   (net +1)
latency >25s (wall risk): [('6785', 'B', 38.3), ('6756', 'B', 29.7), ('6697', 'B', 37.0), ('6927', 'B', 39.4), ('7027', 'B', 39.5), ('6976', 'B', 39.6), ('6744', 'B',